# A graph-walking agent with a judgment ledger

An agent walks the London Underground from Kennington to Euston **one hop at a time**. At every station it can only see its legal moves (`next_hop`), Jev scores them and a margin gate decides who picks: the agent or a person. Every decision is written back into the graph as a `Step` with the options it considered, so afterward you can ask the graph where the agent was unsure.

The store is Relate (formerly CypherLite), the embedded graph database from Neo4j. The database is one file.

**Where Jev runs.** Jev is called directly from Python, on the options Relate supplies and Relate stores the answer. No plugin and no Neo4j instance are involved. It uses the `typesafe-sdk` call. Set `PICKER = "stub"` in section 1 to run offline with a plain scoring rule instead.

**You need** the three Tube CSVs in a `data` folder next to this notebook, Relate installed and `TYPESAFE_API_KEY` set before launching Jupyter.

Section 7 hands the walk to an LLM instead of the scripted loop. Any model can be plugged in (Claude, an OpenAI-compatible API or a local Ollama model) and the model gives a reason with every move.

In [1]:
%pip install pandas==3.1.1 \
             openai==3.49.0 \
             relate-db==1.0.0 \
             typesafe-sdk==0.7.5 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import math
import os
import pandas as pd
import relate
import sys
import uuid

from collections import defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
from typesafe_sdk import Choice, TypeSafeClient

sys.path.append(str(Path("../code").resolve()))   # llm.py lives in the code folder
from llm import make_llm

## 1. Settings

In [3]:
DATA_DIR = next((d for d in (Path("../data"), Path("data")) if d.exists()), Path("../data"))   # the shared data folder
DB_PATH = "walk.cl"       # one file; ":memory:" for a throwaway database
RESET = True              # delete the file first so the notebook can be re-run

ORIGIN, GOAL = "Kennington", "Euston"
LOW_MARGIN = 0.2          # a margin below this hands the choice to a person
MAX_STEPS = 30            # the agent stops after this many hops
MAX_OPTIONS = 6           # the most moves next_hop offers at one station
PICKER = "jev"            # "jev" (hosted, needs TYPESAFE_API_KEY) or "stub" (no key needed)
HOSTED_MODEL = os.environ.get("JEV_MODEL", "jev-1.13.0")

pd.set_option("display.max_colwidth", None)   # show the whole reason in tables

def records(result):
    """Rows as a list of dicts. Uses .data when the result has it, otherwise reads the cursor."""
    if hasattr(result, "data"):
        data = result.data
        return data() if callable(data) else data
    return [dict(row) for row in result]

def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

## 2. Load the Tube into Relate

Load plus two live-status properties on every station: `active` (true) and `delay_minutes` (0). An operator changes those; the agent only reads them.

In [4]:
if RESET and DB_PATH != ":memory:" and os.path.exists(DB_PATH):
    os.remove(DB_PATH)
db = relate.connect(DB_PATH)

stations = pd.read_csv(DATA_DIR / "london_stations.csv")
links = pd.read_csv(DATA_DIR / "london_connections.csv")
lines_csv = pd.read_csv(DATA_DIR / "london_lines.csv")
for frame in (stations, links, lines_csv):
    frame.columns = frame.columns.str.strip()

links = links[links["Tube Line"].str.strip().isin(set(lines_csv["Tube Line"].str.strip()))]
used = set(links["From Station"].str.strip()) | set(links["To Station"].str.strip())
stations = stations[stations["Station"].str.strip().isin(used)]

db.execute(
    "UNWIND $rows AS r "
    "MERGE (s:Station {name: r.name}) "
    "SET s.lat = r.lat, s.lon = r.lon, s.active = true, s.delay_minutes = 0",
    {"rows": [
        {"name": str(r["Station"]).strip(), "lat": float(r["Latitude"]), "lon": float(r["Longitude"])}
        for r in stations.to_dict("records")
    ]},
)
db.execute(
    "UNWIND $rows AS r "
    "MATCH (a:Station {name: r.a}), (b:Station {name: r.b}) "
    "CREATE (a)-[:CONNECTS_TO {line: r.line}]->(b)",
    {"rows": [
        {"a": str(r["From Station"]).strip(), "b": str(r["To Station"]).strip(), "line": str(r["Tube Line"]).strip()}
        for r in links.to_dict("records")
    ]},
)

n_stations = records(db.execute("MATCH (s:Station) RETURN count(s) AS n"))[0]["n"]
n_links = records(db.execute("MATCH ()-[r:CONNECTS_TO]->() RETURN count(r) AS n"))[0]["n"]
print(f"{n_stations} stations, {n_links} connections")

349 stations, 458 connections


## 3. The pickers

A picker takes the state, the question and the options and returns a probability for each option. The stub scores each option with a rule: closer to the goal is better, a suspended station is bad, a suspended station one hop further on is a warning and going back to a station already visited is penalized. It then turns the scores into probabilities with a softmax.

It is a stand-in so the loop runs offline. It is **not** Jev and its margins say nothing about Jev's.

In [5]:
STUB_TEMPERATURE = 1.0
AHEAD_SUSPENDED = 1.5     # penalty when a station one hop further on is suspended
SUSPENDED = 4.0
VISITED = 2.0

def stub_picker(state, questions, options):
    scores = {}
    for o in options:
        s = -1.5 * o["d_goal"]
        if o["suspended"]:
            s -= SUSPENDED
        s -= 0.3 * o["delay"]
        s -= AHEAD_SUSPENDED * len(o["ahead_suspended"]) + 0.2 * sum(m for _, m in o["ahead_delayed"])
        if o["visited"]:
            s -= VISITED
        scores[o["station"]] = s / STUB_TEMPERATURE
    top = max(scores.values())
    exp = {k: math.exp(v - top) for k, v in scores.items()}
    total = sum(exp.values())
    return {k: v / total for k, v in exp.items()}

def scripted_human(pending):
    """Stands in for a person: the closest station to the goal that is open and new, else Jev's top pick."""
    open_new = [o for o in pending["options"] if not o["suspended"] and not o["visited"]]
    if open_new:
        return sorted(open_new, key=lambda o: (o["d_goal"], o["station"]))[0]["station"]
    return pending["top"]

def terminal_human(pending):
    """Asks you at the keyboard."""
    names = [o["station"] for o in pending["options"]]
    print("The margin is low. Options:", ", ".join(f"{n} ({pending['probs'][n]:.2f})" for n in names))
    answer = input("Where should the agent go? ").strip()
    return answer

### The Jev picker

One `Choice` question, the options as `criteria` and the state as plain words. The answer comes back with a probability for each station. `PICKER = "stub"` skips this and uses the stub above.

Hosted decisions are not deterministic, so two walks can differ.

In [6]:
_client = None

def jev_picker(state, questions, options):
    """Ask the hosted Jev which station to go to next. Returns {station: probability}."""
    global _client
    if _client is None:
        if not os.environ.get("TYPESAFE_API_KEY"):
            raise RuntimeError("TYPESAFE_API_KEY isn't set. Set it before launching Jupyter, or use PICKER = 'stub'.")
        _client = TypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"], model=HOSTED_MODEL, timeout=5.0)
    q = questions["hop"]
    response = _client.system_one(
        state=state,
        questions={"hop": Choice(instructions=q["instructions"], criteria=q["criteria"])},
    )
    probabilities = getattr(response.choices["hop"], "probabilities", None)
    if not probabilities:
        raise RuntimeError("Jev returned no probabilities, so there is no margin to judge the hop by.")
    return {k: float(v) for k, v in dict(probabilities).items()}

## 4. The agent's tools

`Walker` holds the six tools, plus the operator tools. The agent never writes Cypher; each tool runs a fixed statement.

- `next_hop` is the only source of legal moves.
- `judge_hop` asks the picker and writes a `Step` with a `CONSIDERED` edge to every option.
- `move` refuses a station that is not a legal move, then records which side chose (`agent` or `human`) and the lines that serve the link it took.
- `ledger_report` runs canned read-only queries.

The distance to the goal is worked out once per walk, with a breadth-first search over the connections and not on every hop.

In [7]:
class Walker:
    def __init__(self, db, origin, goal, picker, human=scripted_human, agent=""):
        self.db, self.goal, self.picker, self.human, self.agent = db, goal, picker, human, agent
        self.walk_id = uuid.uuid4().hex[:8]
        self.here, self.visited, self.n, self.pending = origin, [origin], 0, None
        edges = records(db.execute(
            "MATCH (a:Station)-[r:CONNECTS_TO]->(b:Station) RETURN a.name AS a, b.name AS b"))
        self.adj = defaultdict(set)
        for e in edges:
            self.adj[e["a"]].add(e["b"])
            self.adj[e["b"]].add(e["a"])
        self.dist = self._distances(goal)

    def _distances(self, goal):
        dist, queue = {goal: 0}, deque([goal])
        while queue:
            x = queue.popleft()
            for y in self.adj[x]:
                if y not in dist:
                    dist[y] = dist[x] + 1
                    queue.append(y)
        return dist

    # Operator tools (not given to the agent)
    def set_status(self, station, state, minutes=5):
        active, delay = {"suspended": (False, 0), "delayed": (True, minutes), "clear": (True, 0)}[state]
        self.db.execute(
            "MATCH (s:Station {name: $name}) SET s.active = $active, s.delay_minutes = $delay",
            {"name": station, "active": active, "delay": delay})

    def clear_all(self):
        self.db.execute("MATCH (s:Station) SET s.active = true, s.delay_minutes = 0")

    def _status(self):
        rows = records(self.db.execute(
            "MATCH (s:Station) WHERE s.active = false OR s.delay_minutes > 0 "
            "RETURN s.name AS name, s.active AS active, s.delay_minutes AS delay"))
        return {r["name"]: (r["active"] is False, r["delay"] or 0) for r in rows}

    # Agent tools
    def where_am_i(self):
        return {"station": self.here, "goal": self.goal, "steps": self.n, "visited": list(self.visited)}

    def next_hop(self):
        rows = records(self.db.execute(
            "MATCH (a:Station {name: $here})-[r:CONNECTS_TO]-(b:Station) "
            "RETURN b.name AS station, collect(DISTINCT r.line) AS lines",
            {"here": self.here}))
        status = self._status()
        options = []
        for r in rows:
            st = r["station"]
            if st not in self.dist:
                continue
            off, late = status.get(st, (False, 0))
            ahead = [c for c in self.adj[st] if self.dist.get(c, 99) < self.dist[st] and c in status]
            options.append({
                "station": st, "lines": sorted(r["lines"]), "d_goal": self.dist[st],
                "suspended": off, "delay": late,
                "ahead_suspended": sorted(c for c in ahead if status[c][0]),
                "ahead_delayed": sorted((c, status[c][1]) for c in ahead if not status[c][0] and status[c][1] > 0),
                "visited": st in self.visited,
            })
        options.sort(key=lambda o: (o["d_goal"], o["station"]))
        options = options[:MAX_OPTIONS]
        for o in options:
            o["description"] = self._describe(o)
        return options

    def _describe(self, o):
        lines = ", ".join(o["lines"])
        status = "suspended" if o["suspended"] else (f"delayed {o['delay']} min" if o["delay"] else "clear")
        text = f"{o['station']} ({lines} line). {o['d_goal']} stops from {self.goal}. Status: {status}."
        if o["ahead_suspended"]:
            text += f" Further on: {', '.join(o['ahead_suspended'])} suspended."
        if o["ahead_delayed"]:
            text += " Further on: " + ", ".join(f"{c} delayed {m} min" for c, m in o["ahead_delayed"]) + "."
        if o["visited"]:
            text += " Already visited."
        return text

    def judge_hop(self):
        if self.here == self.goal:
            return {"error": f"Already at {self.goal}."}
        if self.pending:
            return self.pending["result"]   # judged already: return the same answer, don't ask again
        options = self.next_hop()
        if not options:
            return {"error": f"No legal moves from {self.here}."}
        state = {"current_station": self.here, "destination": self.goal,
                 "steps_taken": self.n, "stations_visited": ", ".join(self.visited)}
        questions = {"hop": {
            "type": "choice",
            "instructions": (f"The passenger is at {self.here} and wants to reach {self.goal} as reliably as possible. "
                             "Which station should they go to next?"),
            "criteria": {o["station"]: o["description"] for o in options}}}
        probs = self.picker(state, questions, options)
        ranked = sorted(probs.items(), key=lambda kv: -kv[1])
        top, p1 = ranked[0]
        margin = p1 - (ranked[1][1] if len(ranked) > 1 else 0.0)
        step_id = f"{self.walk_id}-{self.n}"
        self.db.execute(
            "MATCH (st:Station {name: $here}) "
            "CREATE (s:Step {id: $id, walk: $walk, agent: $agent, n: $n, top: $top, probability: $p, margin: $margin, "
            "decided_by: 'pending', created_at: $ts})-[:AT]->(st)",
            {"here": self.here, "id": step_id, "walk": self.walk_id, "agent": self.agent, "n": self.n, "top": top,
             "p": round(p1, 4), "margin": round(margin, 4), "ts": now()})
        self.db.execute(
            "UNWIND $rows AS r "
            "MATCH (s:Step {id: $id}), (b:Station {name: r.name}) "
            "CREATE (s)-[:CONSIDERED {probability: r.p}]->(b)",
            {"id": step_id, "rows": [{"name": k, "p": round(v, 4)} for k, v in probs.items()]})
        result = {"top": top, "probability": round(p1, 3), "margin": round(margin, 3),
                  "needs_human": margin < LOW_MARGIN, "probabilities": {k: round(v, 3) for k, v in ranked}}
        self.pending = {"id": step_id, "top": top, "probs": probs, "margin": margin, "options": options, "result": result}
        return result

    def move(self, station, decided_by="agent", reason=""):
        served = {o["station"]: o["lines"] for o in self.next_hop()}
        legal = list(served)
        if station not in legal:
            return {"ok": False,
                    "error": f"'{station}' is not a legal move from {self.here}. Legal moves: {', '.join(legal)}."}
        if self.pending is None:
            return {"ok": False, "error": "Call judge_hop before move."}
        if decided_by == "agent" and self.pending["margin"] < LOW_MARGIN:
            return {"ok": False, "error": (f"The margin is {self.pending['margin']:.2f}, below {LOW_MARGIN}. "
                                           "Call ask_human so a person chooses.")}
        self.db.execute(
            "MATCH (s:Step {id: $id}), (b:Station {name: $to}) "
            "SET s.decided_by = $by, s.chose = $to, s.reason = $reason, s.lines = $lines "
            "CREATE (s)-[:CHOSE]->(b)",
            {"id": self.pending["id"], "to": station, "by": decided_by, "reason": reason,
             "lines": ", ".join(served[station])})
        self.here, self.pending = station, None
        self.visited.append(station)
        self.n += 1
        result = {"ok": True, "at": station}
        if decided_by == "human":
            result["note"] = (f"A person chose {station} and the walk has moved there. Do not call move. "
                              "Continue with next_hop.")
        return result

    def ask_human(self, reason=""):
        if self.pending is None:
            return {"ok": False, "error": "Call judge_hop before ask_human."}
        why = f"chosen by a person (asked: {reason.strip()})" if reason.strip() else "chosen by a person"
        return self.move(self.human(self.pending), decided_by="human", reason=why)

    def ledger_report(self, kind, low=None, runner_up=0.3):
        low = LOW_MARGIN if low is None else low
        queries = {
            "shaky": (
                "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.margin < $low AND s.chose IS NOT NULL "
                "RETURN s.walk AS walk, s.n AS step, st.name AS station, s.top AS top_pick, "
                "s.chose AS chose, s.margin AS margin, s.decided_by AS decided_by ORDER BY s.margin"),
            "trouble": (
                "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.margin < $low AND s.chose IS NOT NULL "
                "RETURN st.name AS station, count(s) AS shaky_steps ORDER BY shaky_steps DESC, station"),
            "runner_ups": (
                "MATCH (s:Step)-[c:CONSIDERED]->(o:Station) "
                "WHERE c.probability > $runner_up AND o.name <> s.chose "
                "RETURN s.walk AS walk, s.n AS step, o.name AS runner_up, s.chose AS chose, "
                "c.probability AS probability ORDER BY c.probability DESC"),
            "overrides": (
                "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.chose IS NOT NULL AND s.chose <> s.top "
                "RETURN s.walk AS walk, s.n AS step, st.name AS station, s.top AS top_pick, "
                "s.chose AS chose, s.lines AS lines, s.margin AS margin, s.decided_by AS decided_by, "
                "s.agent AS agent, s.reason AS reason ORDER BY s.walk, s.n"),
            "path": (
                "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.chose IS NOT NULL "
                "RETURN s.walk AS walk, s.n AS step, st.name AS at, s.chose AS chose, s.lines AS lines, "
                "s.margin AS margin, s.decided_by AS decided_by, s.reason AS reason ORDER BY s.walk, s.n"),
            "humans": (
                "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.decided_by = 'human' "
                "RETURN s.walk AS walk, s.n AS step, st.name AS station, s.top AS top_pick, "
                "s.chose AS chose, s.margin AS margin"),
        }
        if kind not in queries:
            return {"error": f"Unknown report '{kind}'. Choose from: {', '.join(queries)}."}
        return pd.DataFrame(records(self.db.execute(
            queries[kind], {"low": low, "runner_up": runner_up})))

## 5. The walk

The scripted agent loop is deliberately simple: judge, then either move or ask a person, until it reaches the goal or runs out of steps. An external LLM agent would call the same tools over MCP instead.

In [8]:
def get_picker():
    return jev_picker if PICKER == "jev" else stub_picker

def walk(origin=ORIGIN, goal=GOAL, before_step=None, human=scripted_human):
    w = Walker(db, origin, goal, get_picker(), human)
    rows = []
    while w.here != goal and w.n < MAX_STEPS:
        if before_step:
            before_step(w)
        j = w.judge_hop()
        if "error" in j:
            print(j["error"])
            break
        at, n = w.here, w.n
        r = w.ask_human() if j["needs_human"] else w.move(j["top"])
        if not r["ok"]:
            print(r["error"])
            break
        rows.append({"step": n, "from": at, "to": r["at"], "margin": round(j["margin"], 2),
                     "decided_by": "human" if j["needs_human"] else "agent",
                     "top_pick": j["top"]})
    print(f"{'Reached' if w.here == goal else 'Stopped before'} {goal} after {w.n} hops "
          f"(walk {w.walk_id}).")
    return w, pd.DataFrame(rows)

w1, steps1 = walk()
steps1

Reached Euston after 6 hops (walk 805e6975).


,step,from,to,margin,decided_by,top_pick
0,0,Kennington,Waterloo,0.78,agent,Waterloo
1,1,Waterloo,Westminster,0.64,agent,Westminster
2,2,Westminster,Green Park,0.75,agent,Green Park
3,3,Green Park,Oxford Circus,0.89,agent,Oxford Circus
4,4,Oxford Circus,Warren Street,0.89,agent,Warren Street
5,5,Warren Street,Euston,0.89,agent,Euston


### 5.1 A move the agent made up
`move` checks every request against `next_hop`. A station that is not a neighbor is refused and nothing is written.

In [9]:
probe = Walker(db, ORIGIN, GOAL, get_picker())
probe.judge_hop()
print(probe.move("Atlantis"))
print(probe.move("Euston"))   # on the map, but not a neighbor of the start

{'ok': False, 'error': "'Atlantis' is not a legal move from Kennington. Legal moves: Waterloo, Elephant and Castle, Nine Elms, Oval."}
{'ok': False, 'error': "'Euston' is not a legal move from Kennington. Legal moves: Waterloo, Elephant and Castle, Nine Elms, Oval."}


### 5.2 Disrupt a station mid-walk
After the first hop, the operator suspends a station three stops along the clear-day route. The next `next_hop` shows it as "further on" for the options that lead there and the margin may collapse. The stub penalty is tuned (`AHEAD_SUSPENDED`) so a close call is likely; with the real Jev the outcome is something to observe, not assume.

In [10]:
route = [ORIGIN] + list(steps1["to"])
victim = route[min(3, len(route) - 2)]
print("Clear-day route:", " > ".join(route))
print("Will suspend:", victim)

def disrupt(w):
    if w.n == 1:
        w.set_status(victim, "suspended")

w2, steps2 = walk(before_step=disrupt)
w2.clear_all()
steps2

Clear-day route: Kennington > Waterloo > Westminster > Green Park > Oxford Circus > Warren Street > Euston
Will suspend: Green Park
Reached Euston after 8 hops (walk 62e36682).


,step,from,to,margin,decided_by,top_pick
0,0,Kennington,Waterloo,0.87,agent,Waterloo
1,1,Waterloo,Westminster,0.39,agent,Westminster
2,2,Westminster,Embankment,0.01,human,Embankment
3,3,Embankment,Charing Cross,0.92,agent,Charing Cross
4,4,Charing Cross,Piccadilly Circus,0.70,agent,Piccadilly Circus
5,5,Piccadilly Circus,Oxford Circus,0.92,agent,Oxford Circus
6,6,Oxford Circus,Warren Street,0.93,agent,Warren Street
7,7,Warren Street,Euston,0.83,agent,Euston


## 6. Ask the ledger

Every hop of both walks is in the graph. These four reports are fixed, read-only queries.

In [11]:
for kind in ("shaky", "trouble", "runner_ups", "humans"):
    print(f"--- {kind} ---")
    report = w2.ledger_report(kind)
    display(report if len(report) else "None recorded.")

--- shaky ---


,walk,step,station,top_pick,chose,margin,decided_by
0,62e36682,2,Westminster,Embankment,Embankment,0.01,human


--- trouble ---


,station,shaky_steps
0,Westminster,1


--- runner_ups ---


,walk,step,runner_up,chose,probability
0,62e36682,2,St. James's Park,Embankment,0.36


--- humans ---


,walk,step,station,top_pick,chose,margin
0,62e36682,2,Westminster,Embankment,Embankment,0.01


The same ledger as a path: each step, where it was, what it chose and the options it passed over. Steps that were judged but never taken, like the refused-move probe above, have no `chose` and are left out.

In [12]:
display(pd.DataFrame(records(db.execute(
    "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.chose IS NOT NULL "
    "RETURN s.walk AS walk, s.n AS step, st.name AS at, s.chose AS chose, s.lines AS lines, "
    "s.margin AS margin, s.decided_by AS decided_by, s.reason AS reason ORDER BY s.walk, s.n"))))

,walk,step,at,chose,lines,margin,decided_by,reason
0,62e36682,0,Kennington,Waterloo,Northern,0.87,agent,
1,62e36682,1,Waterloo,Westminster,Jubilee,0.39,agent,
2,62e36682,2,Westminster,Embankment,"Circle, District",0.01,human,chosen by a person
3,62e36682,3,Embankment,Charing Cross,"Bakerloo, Northern",0.92,agent,
4,62e36682,4,Charing Cross,Piccadilly Circus,Bakerloo,0.70,agent,
5,62e36682,5,Piccadilly Circus,Oxford Circus,Bakerloo,0.92,agent,
6,62e36682,6,Oxford Circus,Warren Street,Victoria,0.93,agent,
7,62e36682,7,Warren Street,Euston,"Northern, Victoria",0.83,agent,
8,805e6975,0,Kennington,Waterloo,Northern,0.78,agent,
9,805e6975,1,Waterloo,Westminster,Jubilee,0.64,agent,


## 7. Let an LLM drive the walk

So far a scripted loop played the agent. Here a language model is the agent: it sees the same tools, decides which to call, and writes a one-sentence reason with every move, which goes into the ledger. Jev still scores each hop. It knows nothing of the passenger's constraint, so when a constraint makes a different station better, the model may pick it (if the margin lets it move on its own) and say why.

The model is a setting, not a dependency. `llm.py` (in the `code` folder, which the next cell finds on its own) holds one small adapter per provider: `anthropic:<model>`, `openai:<model>` (any OpenAI-compatible API) and `ollama:<model>`. Install the SDK you use (`pip install anthropic`, or `pip install openai` for the other two) and set its key (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`; Ollama needs none). To add another provider, write one class in `llm.py` and register it.

The model matters most when there is a constraint. In our runs a cheaper model (`gpt-6-luna`) handled no constraint and the Jubilee constraint but read the line lists inconsistently on "avoid the Northern line" and wandered for 16 to 18 hops, while `gpt-6.1-terra` took the direct route on all three. The ledger shows the difference: the `agent` column says which model chose, and the `overrides` report shows each departure from Jev with the reason given.

In [13]:
LLM = "openai:gpt-6.1-terra" # provider:model, e.g. "openai:gpt-6-luna" (cheaper, weaker on constraints). For a local model: "ollama:bonsai-2:9b"
CONSTRAINT = None            # what the passenger wants, in words. Try None, "avoid the Northern line" (every hop out of
                             # Kennington is Northern, so one hop is unavoidable) or "avoid the Jubilee line" (conflicts with Jev's top pick)
HUMAN = scripted_human       # or terminal_human, to be asked at the keyboard when the margin is low

In [14]:
NO_ARGS = {"type": "object", "properties": {}}
AGENT_TOOLS = [
    {"name": "where_am_i", "description": "Where the walk is: current station, goal, hops taken, stations visited.",
     "parameters": NO_ARGS},
    {"name": "next_hop", "description": "The legal moves from the current station, with lines, stops to the goal, live status "
     "and any trouble one hop further on. The only list of legal moves.", "parameters": NO_ARGS},
    {"name": "judge_hop", "description": "Ask Jev to score the legal moves and write the decision to the ledger. Returns Jev's "
     "top pick, a probability for each move, the margin, and needs_human.", "parameters": NO_ARGS},
    {"name": "move", "description": "Move to one of the legal stations from next_hop, after judge_hop. Refused if the station "
     "is not legal or if needs_human was true. Give a one-sentence reason; it is stored in the ledger.",
     "parameters": {"type": "object", "properties": {"station": {"type": "string"}, "reason": {"type": "string"}},
                    "required": ["station"]}},
    {"name": "ask_human", "description": "Ask a person which station to go to. The person's choice is made as the move, so you are moved "
     "there: do not call move afterward. Use it when judge_hop says needs_human, or when every legal move breaks the passenger's "
     "constraint. If some legal move meets the constraint, use move instead. Give a one-sentence reason; it is stored in the ledger.",
     "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}}},
]

SYSTEM = """You are walking a passenger across the London Underground one hop at a time, from a start station to a goal.
You cannot see the map. Only next_hop lists your legal moves. Loop until the goal is reached:
1. next_hop, 2. judge_hop (Jev scores the moves), 3. if needs_human is true call ask_human, otherwise call move.
If needs_human is true, ask_human makes the move for you: do not call move afterward, continue with next_hop.
Call ask_human for the passenger's constraint only when every legal move breaks it, and give the reason. If any legal move
meets the constraint, choose it yourself with move (when the margin allows) and say why.
A move meets a constraint to avoid a line if at least one of the lines listed for it is not that line, because the passenger can ride that other line. Judge every move this way, every time.
Never invent a station. Give a one-sentence reason with every move; it is stored in the ledger.
Base every reason (including one you give to ask_human) only on what next_hop and judge_hop returned (lines, status, probabilities). Say "probability", not
"reliability score", and do not call a move the only option unless next_hop lists just one. A probability is Jev's chance that a station is the best next move, not a chance of reaching the goal.
Jev scores each hop for reliability only and does not know the passenger's constraints. If a constraint means a different
legal station is better than Jev's top pick, you may choose it, as long as the margin lets you move yourself, and say why.
Stop as soon as the goal is reached."""

def run_tool(w, name, args):
    if name == "where_am_i":
        return w.where_am_i()
    if name == "next_hop":
        return {"at": w.here, "goal": w.goal, "moves": w.next_hop()}
    if name == "judge_hop":
        return w.judge_hop()
    if name == "move":
        return w.move(args["station"], reason=args.get("reason", ""))
    if name == "ask_human":
        return w.ask_human(args.get("reason", ""))
    return {"error": f"Unknown tool '{name}'."}

def llm_walk(llm_spec=LLM, constraint=CONSTRAINT, origin=ORIGIN, goal=GOAL, before_step=None, max_turns=80):
    llm = make_llm(llm_spec)
    w = Walker(db, origin, goal, get_picker(), human=HUMAN, agent=llm.label)
    task = f"Walk the passenger from {origin} to {goal}."
    if constraint:
        task += f" The passenger's constraint: {constraint}."
    messages = [{"role": "user", "content": task}]
    print(f"Walk {w.walk_id}, driven by {llm.label}\n")
    for _ in range(max_turns):
        if before_step:
            before_step(w)
        reply = llm.chat(SYSTEM, messages, AGENT_TOOLS)
        messages.append({"role": "assistant", "content": reply.text, "tool_calls": reply.tool_calls})
        if reply.text:
            print(f"[{llm.label}] {reply.text.strip()}")
        if not reply.tool_calls:
            break
        for c in reply.tool_calls:
            out = run_tool(w, c.name, c.arguments)
            messages.append({"role": "tool", "tool_call_id": c.id, "name": c.name, "content": json.dumps(out, default=str)})
            if c.name == "judge_hop" and "margin" in out:
                print(f"  judge_hop: Jev favors {out['top']} (margin {out['margin']:.2f})"
                      f"{'  -> needs a person' if out['needs_human'] else ''}")
            elif c.name in ("move", "ask_human"):
                why = f"  ({c.arguments['reason']})" if c.arguments.get("reason") else ""
                print(f"  {c.name}: -> {out['at']}{why}" if out.get("ok") else f"  {c.name}: refused. {out.get('error')}")
            else:
                print(f"  {c.name}")
        if w.here == goal:
            break
    print(f"\n{'Reached' if w.here == goal else 'Stopped before'} {goal} after {w.n} hops (walk {w.walk_id}).")
    return w

Run it with a station suspended after the first hop, as in section 5. The `disrupt` function from there is reused.

In [15]:
w3 = llm_walk(before_step=disrupt)
w3.clear_all()

Walk f473e46c, driven by openai:gpt-6.1-terra

  next_hop
  judge_hop: Jev favors Waterloo (margin 0.81)
  move: -> Waterloo  (Waterloo is clear and has Jev's highest probability at 0.90.)
  next_hop
  judge_hop: Jev favors Westminster (margin 0.43)
  move: -> Westminster  (Westminster is clear and has Jev's highest probability at 0.63, though Green Park is suspended further on.)
  next_hop
  judge_hop: Jev favors St. James's Park (margin 0.02)  -> needs a person
  ask_human: -> Embankment  (Jev's margin is 0.02 between clear St. James's Park and Embankment, so a human choice is required.)
  next_hop
  judge_hop: Jev favors Charing Cross (margin 0.91)
  move: -> Charing Cross  (Charing Cross is clear and has Jev's highest probability at 0.94.)
  next_hop
  judge_hop: Jev favors Piccadilly Circus (margin 0.69)
  move: -> Piccadilly Circus  (Piccadilly Circus is clear and has Jev's highest probability at 0.83.)
  next_hop
  judge_hop: Jev favors Oxford Circus (margin 0.92)
  move: -> Oxf

The path the model took, with its reasons, and every step where the choice differed from Jev's top pick. `decided_by` says whether the model or a person chose.

In [16]:
display(pd.DataFrame(records(db.execute(
    "MATCH (s:Step)-[:AT]->(st:Station) WHERE s.walk = $w AND s.chose IS NOT NULL "
    "RETURN s.n AS step, st.name AS at, s.chose AS chose, s.lines AS lines, s.margin AS margin, s.decided_by AS decided_by, "
    "s.reason AS reason ORDER BY s.n", {"w": w3.walk_id}))))
overrides = w3.ledger_report("overrides")
print("Where a choice differed from Jev's top pick (all walks):")
display(overrides if len(overrides) else "No overrides recorded.")

,step,at,chose,lines,margin,decided_by,reason
0,0,Kennington,Waterloo,Northern,0.81,agent,Waterloo is clear and has Jev's highest probability at 0.90.
1,1,Waterloo,Westminster,Jubilee,0.43,agent,"Westminster is clear and has Jev's highest probability at 0.63, though Green Park is suspended further on."
2,2,Westminster,Embankment,"Circle, District",0.02,human,"chosen by a person (asked: Jev's margin is 0.02 between clear St. James's Park and Embankment, so a human choice is required.)"
3,3,Embankment,Charing Cross,"Bakerloo, Northern",0.91,agent,Charing Cross is clear and has Jev's highest probability at 0.94.
4,4,Charing Cross,Piccadilly Circus,Bakerloo,0.69,agent,Piccadilly Circus is clear and has Jev's highest probability at 0.83.
5,5,Piccadilly Circus,Oxford Circus,Bakerloo,0.92,agent,Oxford Circus is clear and has Jev's highest probability at 0.95.
6,6,Oxford Circus,Warren Street,Victoria,0.92,agent,Warren Street is clear and has Jev's highest probability at 0.94.
7,7,Warren Street,Euston,"Northern, Victoria",0.83,agent,Euston is clear and has Jev's highest probability at 0.91.


Where a choice differed from Jev's top pick (all walks):


,walk,step,station,top_pick,chose,lines,margin,decided_by,agent,reason
0,f473e46c,2,Westminster,St. James's Park,Embankment,"Circle, District",0.02,human,openai:gpt-6.1-terra,"chosen by a person (asked: Jev's margin is 0.02 between clear St. James's Park and Embankment, so a human choice is required.)"


In [17]:
db.close()
print(f"Closed. The ledger is in {DB_PATH} ({os.path.getsize(DB_PATH) / 1024:.0f} KB)." if os.path.exists(DB_PATH) else "Closed.")

Closed. The ledger is in walk.cl (400 KB).
